# AVA v3-Code — resumable training notebook (Colab / Kaggle / laptop)

**One loop, three platforms, preemption-safe.** Every platform is a disposable worker.
The canonical state lives in a Hugging Face Hub repo. Any session can die; re-run this
notebook anywhere and it resumes from the last push within one sync interval.

Pattern (from `scripts/checkpoint_sync.py`):
- checkpoint = **trainable params + optimizer + RNG + step**, pushed every `SYNC_MINUTES`
- `LATEST.json` is uploaded **last** → a torn upload never corrupts resume
- data order is deterministic from `seed + step` → we fast-forward the stream on resume
- each session is **time-boxed** to `SHARD_MINUTES` (< platform limit), then force-saves and exits

Phases (CODE_PIVOT.md §8): **C1** donor baseline → **C2** linearize → **C3** MoE/ternary →
**C4** mount HRM → **C5** SFT → **C6** self-play RL → **C7** pack → **C8** eval/release.
This notebook runs **C1 (donor eval)** and a generic **LoRA shard trainer** you point at any phase's data.

> ⚠️ Read the review doc `docs/v3/REVIEW_2026-07.md` first — it changes what C2/C3 should be.

## 0. Setup

In [ ]:
%pip install -U -q "transformers>=4.56" "datasets>=2.20" "accelerate>=0.33"     "peft>=0.12" "bitsandbytes>=0.46.1" "huggingface_hub>=0.24" pyyaml
# best-effort fast kernels for Qwen3.5's Gated DeltaNet layers (large speedup;
# torch fallback works if these fail to install)
!pip install -q flash-linear-attention causal-conv1d 2>&1 | tail -1 || true
import bitsandbytes, transformers
print('deps ready | bitsandbytes', bitsandbytes.__version__,
      '| transformers', transformers.__version__)

## 1. Auth — HF token (never hard-code it)
- **Kaggle**: Add-ons → Secrets → add `HF_TOKEN`.
- **Colab**: 🔑 sidebar → add secret `HF_TOKEN`.
- **Laptop**: `export HF_TOKEN=...` or `huggingface-cli login` once.

In [ ]:
import os
tok = os.environ.get('HF_TOKEN')
if not tok:
    try:  # Kaggle
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not tok:
    try:  # Colab
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
    except Exception:
        pass
assert tok, 'Set HF_TOKEN (secret/env). It needs WRITE on your checkpoint repo.'
os.environ['HF_TOKEN'] = tok
from huggingface_hub import login; login(tok, add_to_git_credential=False)
print('hf auth ok')

## 2. Platform + GPU

In [ ]:
import torch
def where():
    import os
    if os.path.exists('/kaggle'): return 'kaggle'
    if 'COLAB_GPU' in os.environ or os.path.exists('/content'): return 'colab'
    return 'local'
PLATFORM = where()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
vram = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
print(f'platform={PLATFORM}  gpu={gpu}  vram={vram:.1f}GB  torch={torch.__version__}')
# Batch size / seq len / shard cadence are chosen automatically per GPU by
# train/hw_profile.py at run time — nothing to configure here.

## 3. Get the repo + CheckpointSync
Set `REPO_URL` to your AVA repo (git). We import the tested fabric rather than re-implement it.

In [ ]:
import os, pathlib, sys
REPO_URL = os.environ.get('AVA_REPO_URL', 'https://github.com/NAME0x0/AVA.git')
ROOT = pathlib.Path('/kaggle/working' if PLATFORM=='kaggle' else '/content' if PLATFORM=='colab' else '.')
repo = ROOT / 'AVA'
if not repo.exists():
    # public repo: plain clone. Private repo: set a GITHUB_TOKEN secret (a
    # GitHub PAT — NOT the HF token) and it gets embedded for auth.
    gh_tok = os.environ.get('GITHUB_TOKEN')
    if not gh_tok:
        try:
            from google.colab import userdata
            gh_tok = userdata.get('GITHUB_TOKEN')
        except Exception:
            gh_tok = None
    url = REPO_URL.replace('https://', f'https://{gh_tok}@') if gh_tok else REPO_URL
    rc = os.system(f'git clone --depth 1 {url} {repo}')
    assert rc == 0, 'git clone failed — check repo visibility / GITHUB_TOKEN secret'
scripts = repo / 'experiments' / 'exp6_v3' / 'scripts'
assert scripts.exists(), f'not found: {scripts} — fix REPO_URL'
sys.path.insert(0, str(scripts.parent))  # exp6_v3 root -> scripts.*, train.*, engine.*
from scripts.checkpoint_sync import CheckpointSync
print('repo ready at', repo)

## 4. Autopilot — phase selected from Hub progress, nothing to edit
`train/phase_controller.py` reads the checkpoint repo and picks the next step:
**C1** (baseline; resumes missing eval modes) → **C5** (SFT shards until
total_steps) → **C5_EVAL** (candidate eval + ≤2pp gate vs baseline) → **DONE**.
Press Run-all on any platform; the pipeline always advances.
Override: `AVA_PHASE` env var.

In [ ]:
import os, random
import numpy as np
import yaml
CKPT_REPO = 'NAME0x0/AVA-v3-checkpoints'   # private Hub repo; auto-created
DONOR     = 'Qwen/Qwen3.5-4B'               # REVIEW_2026-07.md section 10
SEED      = 1234

from train.phase_controller import decide_phase, describe
TOTAL_STEPS = yaml.safe_load(
    (repo / 'experiments/exp6_v3/configs/v30_sft.yaml').read_text())['total_steps']
DECISION = decide_phase(CKPT_REPO, TOTAL_STEPS)
PHASE = os.environ.get('AVA_PHASE') or DECISION.phase
print(describe(DECISION))
if os.environ.get('AVA_PHASE'):
    print(f'  (overridden by AVA_PHASE -> {PHASE})')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 5. C1 — donor baseline (the bar; run once per mode on Kaggle)
Full HumanEval+ / MBPP+ executed in the sandbox + ARC-E/MMLU floors, both
thinking modes. Logic lives in `train/c1_eval.py` (tested repo code). Report
JSON pushed to the checkpoint repo — `train/gate.py` compares every later
phase against it. Skip for training runs.

In [ ]:
if PHASE == 'C1':
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from train.c1_eval import run_c1

    # dtype safety: Colab/Kaggle T4 (Turing) has NO bf16 — fp16 there, bf16 on L4/A100
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=dtype,
                             bnb_4bit_use_double_quant=True)
    tok_ = AutoTokenizer.from_pretrained(DONOR, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(DONOR, quantization_config=bnb,
                                                 device_map='auto', trust_remote_code=True,
                                                 dtype=dtype)
    report = run_c1(model, tok_, out_path='c1_donor_baseline.json',
                    modes=DECISION.modes_needed or (False, True),
                    code_limit=None,               # full HumanEval+/MBPP+
                    hub_repo=CKPT_REPO)
    for mode, r in report.items():
        if mode != 'meta':
            print(mode, {k: v['score'] for k, v in r.items()})

## 6. Training shard (C5 lane) — config-driven, resumable
All logic in `train/sft.py` + `configs/v30_sft.yaml` (both under test).
Per-source mixture cursor rides inside each checkpoint -> resume anywhere
restores the exact data position. Edit the YAML, not this cell.

In [ ]:
if PHASE != 'C1':
    from train.sft import SFTConfig, train_shard

    cfg = SFTConfig.from_yaml(str(repo / 'experiments/exp6_v3/configs/v30_sft.yaml'))
    cfg.phase = PHASE
    cfg.ckpt_repo = CKPT_REPO
    cfg.donor = DONOR
    # auto_hardware=True (default): train_shard detects the GPU (T4/L4/A100/
    # V100/A2000/CPU) and sets seq_len, grad_accum, shard/sync cadence + the
    # right compute dtype (fp16 on T4/V100, bf16 on L4/A100). Works identically
    # on free tier, compute units, Pro, or Kaggle. P100 and TPU are rejected
    # with a clear error — switch the accelerator instead.
    # Manual override: set cfg.auto_hardware = False and edit fields yourself.
    summary = train_shard(cfg)
    print('shard summary:', summary)

## 6a. C5_EVAL — candidate eval + gate (automatic when training completes)
Loads donor + trained adapters from the Hub, runs the same eval as C1, pushes
`reports/c5_candidate_eval.json`, then prints the ≤2pp gate verdict.

In [ ]:
if PHASE == 'C5_EVAL':
    import sys, torch
    sys.path.insert(0, str(repo / 'experiments' / 'exp6_v3'))
    from train.sft import SFTConfig, _build_qlora_model
    from train.hw_profile import detect_profile
    from train.c1_eval import run_c1
    from train.phase_controller import gate_candidate
    from scripts.checkpoint_sync import CheckpointSync

    cfg = SFTConfig.from_yaml(str(repo / 'experiments/exp6_v3/configs/v30_sft.yaml'))
    cfg.ckpt_repo, cfg.donor = CKPT_REPO, DONOR
    prof = detect_profile()
    dt = torch.bfloat16 if prof.compute_dtype == 'bfloat16' else torch.float16
    model, tok_ = _build_qlora_model(cfg, dt)
    CheckpointSync(CKPT_REPO, phase='C5', trainable_only=True).resume(model)
    model.eval()
    run_c1(model, tok_, out_path='c5_candidate_eval.json',
           modes=(False, True), code_limit=None, hub_repo=CKPT_REPO)
    verdict = gate_candidate(CKPT_REPO)
    print('GATE PASSED' if verdict.passed else 'GATE FAILED')
    print('deltas vs donor:', verdict.deltas)
    for f in verdict.failures:
        print('  FAIL:', f)

if PHASE == 'DONE':
    print(DECISION.reason)

## 6b. Which runtime to pick (free / compute units / Pro) — cheat sheet

Same notebook, same checkpoints, any tier. Resume is automatic in all cases
(HF Hub `LATEST.json`; max loss = one sync interval).

| Tier | What you get | What to click | Notes |
|---|---|---|---|
| **Colab free** | T4, ~15-30 h/wk, preemptible | Runtime > T4 GPU | fp16 auto-selected; shards 150 min |
| **Colab Pay-As-You-Go** | 100 CU / $9.99, no expiry sub | **T4 for steady work, L4 when offered** | CU burn: T4 lowest, L4 ~2x T4 cost but ~2-3x speed (better per-CU), A100 ~6-10x cost (burst only) |
| **Colab Pro ($9.99/mo)** | 100 CU/mo + longer sessions **+ 15 extra Kaggle GPU-h/wk** | L4 default, T4 to stretch CUs | the Kaggle bonus makes Pro strictly better than one-off CU purchase at same price |
| **Kaggle free** | 30 h/wk | **Always GPU T4 x2** | never P100 (bitsandbytes needs compute cap >= 7.0; P100 = 6.0 — trainer rejects it) |
| **TPU (any)** | — | **never** | PyTorch+bitsandbytes = CUDA only; trainer rejects with clear error |

Rules of thumb: L4 = best value when available. A100 = only when you need a
result today (CU burn outpaces its speedup). TPU/P100 = auto-rejected, not a
footgun. Buying CUs mid-run is safe: session dies at 0 CU, checkpoint fabric
loses <= sync interval, resume on free tier continues the same run.

## 7. Resume anywhere
Nothing to configure. Re-open this notebook on **any** platform, run cells 0–4 and 6,
call `train()`. It pulls `LATEST.json` for `PHASE`, restores trainable weights + optimizer
+ RNG, fast-forwards the stream, and keeps going. Kaggle out of quota → switch to Colab →
switch to laptop overnight. Same repo, same run.

**Artifacts** (all in `NAME0x0/AVA-v3-checkpoints`):
`checkpoints/<phase>/step<N>/state.pt` + `checkpoints/<phase>/LATEST.json`.
At each phase gate, export the winning checkpoint to `NAME0x0/AVA-v3` (BF16) and later a GGUF build.